# Aksara OCR — backbone comparison on Kaggle

Trains the architecture-comparison table the paper still needs: several CNN and
transformer backbones on the unified 895-class task. Everything else
(pipeline, resume, verify) is identical to the ablation notebook.

## Run with Save & Run All (batch), not interactively
Interactive Kaggle sessions die at ~40 min idle. Use **Save Version → Save &
Run All (Commit)** so it runs headless. Resume across commits: **+ Add Input →
Your Work →** the previous output, then Save & Run All again.

## Setup (right sidebar → Settings)
1. **Accelerator → GPU T4 x2** (P100 is sm_60, unsupported by Kaggle's torch).
2. **Internet → On** (clone, pip, Mendeley fetch).

## Cost — plan for two sessions
- **CNNs @64px** (`backbone_cnn.yaml`): 15 runs, resnet18 restored from the size
  ablation, so ~12 actually train (~25 min each ≈ 5 h). One session.
- **Transformers @224px** (`backbone_transformer.yaml`): 6 runs (~85 min each ≈
  8–9 h). One session.
Transformers are fixed at 224px; CNNs run at 64px where the size ablation showed
they are near their ceiling. resnet18 has both 64px and 224px numbers, so it
bridges the two groups in the paper — report the resolution difference plainly.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Settings (right sidebar) > Accelerator > GPU T4 x2, then rerun."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]

print(f"{name}  ({arch})")
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}  supports: {supported}")

# torch.cuda.is_available() returns True even when this build ships no kernels
# for the device - the failure then surfaces as a warning storm with every run
# landing in failures.jsonl. Check the architecture explicitly and stop here.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, but this PyTorch build only has kernels for "
        f"{supported}. Switch Settings > Accelerator to GPU T4 x2 (sm_75) "
        f"and rerun. The P100 is sm_60 and will not work."
    )

# Prove a real kernel runs, not just that a device is listed.
probe = (torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")).sum()
torch.cuda.synchronize()
print(f"GPU compute OK (probe={probe.item():.1f})")

In [ ]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

In [ ]:
# Resume: copy finished runs into the working results dir so the runner skips
# them. Sources, all safe no-ops when empty:
#   - committed size/aug ablations in the clone -> restores resnet18@64
#   - committed backbone results in the clone   -> restores earlier commits
#   - an attached previous version's output
# A run is "done" when its result.json exists; that is all the runner checks.
import shutil
from pathlib import Path

ARTIFACTS = Path("/kaggle/working/artifacts")
RESULTS = ARTIFACTS / "results" / "backbones"
RESULTS.mkdir(parents=True, exist_ok=True)

sources  = list(REPO.glob("artifacts_kaggle/**/results/ablations"))   # size/aug ablations
sources += list(REPO.glob("artifacts_kaggle/**/results/backbones"))   # prior backbone commits
sources += list(Path("/kaggle/input").glob("*/artifacts/results/*"))  # attached output

restored = 0
for cand in sources:
    if not cand.is_dir():
        continue
    for run_dir in cand.iterdir():
        # Only unified-task runs are relevant here; skip per_script/script_id.
        if not run_dir.is_dir() or not (run_dir / "result.json").exists():
            continue
        if "__unified__" not in run_dir.name:
            continue
        dest = RESULTS / run_dir.name
        if not dest.exists():
            shutil.copytree(run_dir, dest); restored += 1

done = len(list(RESULTS.glob("*/result.json")))
print(f"restored {restored} finished unified run(s); {done} in the working dir")
print("the runner skips these and trains only the backbones still missing")

In [ ]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

In [ ]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

In [ ]:
# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

In [ ]:
# CNN backbones @64px. resnet18 is restored above and skips; the other four
# train. --num-workers 4 (Kaggle vCPUs); --time-budget 8 stops cleanly before
# the batch limit — whatever finished is saved, attach it next commit to resume.
!python scripts/02_run_matrix.py --config configs/backbone_cnn.yaml \
    --artifacts "{ARTIFACTS}" --results "{RESULTS}" \
    --num-workers 4 --time-budget 8

In [ ]:
# Transformer backbones @224px — the expensive half (~85 min/run). If the CNN
# cell used most of the budget, this may not finish; that is fine, it resumes.
# Consider running this in its own session/commit.
!python scripts/02_run_matrix.py --config configs/backbone_transformer.yaml \
    --artifacts "{ARTIFACTS}" --results "{RESULTS}" \
    --num-workers 4 --time-budget 8

In [ ]:
# Backbone comparison table (whatever has finished so far).
import pandas as pd
from pathlib import Path
import json

rows = []
for f in Path(RESULTS).glob("*/result.json"):
    d = json.load(open(f)); e = d["experiment"]; m = d["test_metrics"]
    rows.append({"model": e["model"], "size": e["image_size"], "seed": e["seed"],
                 "accuracy": m["accuracy"]*100, "macro_f1": m["macro_f1"]*100,
                 "params_M": round(d["num_params"]/1e6, 2)})
df = pd.DataFrame(rows)
if len(df):
    agg = (df.groupby(["model","size","params_M"])
             .agg(acc=("accuracy","mean"), acc_sd=("accuracy","std"),
                  mF1=("macro_f1","mean"), mF1_sd=("macro_f1","std"),
                  seeds=("seed","nunique")).reset_index()
             .sort_values("mF1", ascending=False))
    pd.set_option("display.width", 200)
    print(agg.to_string(index=False))
else:
    print("no backbone runs finished yet")

## After the run
Output is saved automatically with the committed version. If not everything
finished, start a new version, **+ Add Input → Your Work →** this output, and
**Save & Run All** again — the restore cell brings finished runs back and the
rest continues. Back the metrics up to git as before (result.json + report,
not the npz/npy).